# Sensitivity grid — a different six anchors

The same six calibrations, `birt` -> `bmi` -> `flat` in each, on the `_Aug2026_alt` design files.
**Generated by `notebooks/make_design_notebooks.py` — edit the generator,
not this file.** All logic is in `mice_imp/run_grid.py`.

## What this varies — read this before quoting any number from it

The `_alt` files keep the main delivery's design structure exactly — same 1363
respondents in the same order, same form assignment, same anchor count, same
form-pair overlap and items per form — and change **which items are the
anchors**:

| | anchors |
|---|---|
| main | `DA054` `DA058` `DA060` `DA064` `DA066` `DA067` |
| alt | `DA052` `DA058` `DA066` `DA073` `Bike_3` `Bike_7` |

Four of the six are replaced, and the single-form items are reshuffled across
forms to make room. Observed responses agree with the main files cell for cell
wherever both carry one, so this is a different mask over the same data.
Verified against the delivered files on 2026-09-14. It is a sensitivity
analysis on **anchor choice**, as `reduced_anchor` is on anchor count.

Issue B bites the new set directly: `Bike_7` is a member of the strongest
dependent pair on the scale (`Bike_1` / `Bike_7`, r = 0.931), and `DA073`
correlates 0.888 with `DA074`.

### `nobike` is not like the other `nobike` runs

Two of the six anchors are bicycling items. Dropping the bike block therefore
removes a third of the linking as well as the block itself — unlike `main` and
`reduced_anchor`, whose anchors are all `DA*`:

| design | overlap 1&2 / 1&3 / 2&3 — alt `full` | alt `nobike` |
|---|---|---|
| CommonItem | 6 / 6 / 6 | **4 / 4 / 4** |
| Binterlaced | 2 / 2 / 2 | **2 / 2 / 0** |
| Standard | 3 / 3 / 0 | **3 / 1 / 0** |

Binterlaced `nobike` ends up with forms 2 and 3 sharing **no item**, as
Standard has by design. Both item sets are run (decided 2026-09-14), but in
this variant the `full`-vs-`nobike` contrast confounds dropping the bike block
with weakening the linking. Say so wherever it is reported.

### Preflight check 2b FAILs here too

Run on 2026-09-14: **9–10 items have a category with fewer than 20 responses**
(`DA061`, `DA053`, `DA068`, `Bike_1`, `Bike_6`, `DA069_r`, `DA074`, `DA067`,
`DA060`, and `DA066` in Standard and Binterlaced) — about the same as the main
files. This is the standing, deferred issue E; running this grid does not
resolve it.

Order is CommonItem, Binterlaced, Standard, and `full` before `nobike`:
increasing difficulty, so a failure lands where there is the most signal to
diagnose it. `standard/nobike` is the thinnest cell in the grid and runs last.

The runner is **resumable** — a run whose three arm `.npz` files already
exist is skipped, so an interrupted weekend continues rather than restarting
— **isolated**, in that one run raising does not stop the other five, and
**logged**, with each run's output captured to
`results_alt_anchor/<design>/<itemset>/run.log` and `results_alt_anchor/grid_summary.csv`
rewritten after every run.

Everything this notebook writes goes under **`results_alt_anchor/`**, and every
exported CSV row carries `variant = "alt_anchor"`, so these estimates cannot be
pooled with the other grid's by accident.

## For a weekend run, prefer the terminal

A notebook kernel is the wrong container for something that has to survive
until Monday: a closed laptop, a suspended session or a dropped connection
takes the kernel and the run with it. From a terminal instead:

```
nohup python -m mice_imp.run_grid --variant alt_anchor > results_alt_anchor/grid.out 2>&1 &
tail -f results_alt_anchor/grid.out
```

Same script, same resumability. Use the cells below to watch progress, or to
run a single cell of the grid interactively.

## Smoke first

Run the wiring check before committing a weekend to a full pass — it is
2 chains of 50/50 with a
20-epoch ADVI warm-start, against 4 x
3000/2000 for the real thing. It writes to the same
directories, so a real run afterwards needs `--force` (or delete the smoke
`.npz` files).

It is sized to answer *does the plumbing work*, not *what is the answer*: at
50 warmup draws the R-hats and the noise floor mean nothing.
Read only the three checks below off it. The floor on how fast it can go is
the B-MI imputation fit, which does not shrink with chain length; a re-run in
a directory that already has `imputation_model/` reuses it.

Override any of it with `--num-chains`, `--num-warmup`, `--num-samples`,
`--advi-iterations`.

Three things to read off it, each of which silently invalidates a run:

- `Initializing from ADVI solution` must appear for **all three** arms. If it
  appears for some and not others, the `flat`-vs-`birt` gap is measuring
  initialisation rather than Monte Carlo noise, and the noise floor is
  meaningless (CLAUDE.md fact 17).
- the **B-MI flat-fallback rate** must be low. Every fallback downgrades that
  cell to B-IRT, so a high rate means B-MI is not really running (fact 3).
- `flat` and `birt` must land on top of each other. They target the same
  posterior by construction; a systematic gap means the plumbing is wrong
  (fact 2).

In [1]:
import subprocess, sys
from pathlib import Path

REPO = Path("/home/luis/CGE/mice_imp")

def run_grid(*flags):
    """Run mice_imp.run_grid as a subprocess, streaming its output."""
    cmd = [sys.executable, "-m", "mice_imp.run_grid", *flags]
    print(" ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()


# Wiring check only -- uncomment to run it. It writes .npz files, so the
# full pass below would then need --force.
# run_grid("--smoke", "--variant", "alt_anchor")

## The full grid

4 chains, 3000 warmup, 2000 samples, six times over. Only start this once
the smoke output above looks right.

No `--force` here: `results_alt_anchor/` starts empty, so nothing is
skipped. Add it only after a smoke run has put `.npz` files there.

In [2]:
run_grid("--variant", "alt_anchor")

/home/luis/.cache/pypoetry/virtualenvs/mice-imp-toZTZ-oV-py3.12/bin/python -m mice_imp.run_grid --variant alt_anchor


grid: 6 run(s), variant=alt_anchor (data *_Aug2026_alt.dta -> results_alt_anchor/), smoke=False, force=False, 4 chains 3000/2000 advi 100
  -> skipped (already complete)  (0.0 min)  summary: /home/luis/CGE/mice_imp/results_alt_anchor/grid_summary.csv
  -> skipped (already complete)  (0.0 min)  summary: /home/luis/CGE/mice_imp/results_alt_anchor/grid_summary.csv

Binterlaced / full  [alt_anchor]  (27 items, 4 chains, 3000/2000, advi 100)
  data: WDFAB_ExperimentalData_CMBinterlaced_Aug2026_alt.dta
  log: /home/luis/CGE/mice_imp/results_alt_anchor/binterlaced/full/run.log



=== arm: birt ===
compat: patched bayesianquilts.util.training_loop for the batchless marginal-ADVI call (see mice_imp/compat.py).
    ADVI warm-start (100 iterations)
Marginal ADVI (Rao-Blackwellized abilities)
  Item vars: ['difficulties0', 'discriminations', 'ddifficulties']
  Surrogate params: 324
  Rank: 0 (mean-field)


--- Starting Training ---
Patience for early stopping: 3 epochs
LR decay factor on plateau: 0.5
Convergence will be checked every: 1 epoch(s)
Checkpoints will be saved to: None
Optimizing keys: ['mu\\identity\\normal\\loc', 'mu\\identity\\normal\\scale', 'difficulties0\\identity\\normal\\loc', 'difficulties0\\identity\\normal\\scale', 'discriminations\\softplus\\normal\\loc', 'discriminations\\softplus\\normal\\scale', 'ddifficulties\\softplus\\normal\\loc', 'ddifficulties\\softplus\\normal\\scale']
-------------------------
  -> New best loss found. Checkpoint saved.                    
  -> New best loss found. Checkpoint saved.                    
  -> New best loss found. Checkpoint saved.                    
  -> New best loss found. Checkpoint saved.                    
  -> New best loss found. Checkpoint saved.                    
  -> New best loss found. Checkpoint saved.                    
  -> New best loss found. Checkpoint saved.                    
  -> New best loss fo

  Initializing from ADVI solution
  Flat parameter vector: 135 elements
  Mass matrix: diagonal

  Chain 1/4: window_adaptation warmup


    warmup done. adapted step_size=1.2502e-03
    Sampling (2000 steps → 2000 kept, step_size=0.001250)...


... (158 outputs left out)

    Non-divergent ratio: 0.060



  Kept 4/4 chains
  Mean non-divergent: 0.057
  difficulties0: mean=-0.3871, std=0.5544
  discriminations: mean=2.1154, std=0.9705
  ddifficulties: mean=0.8085, std=0.3886
    R-hat difficulties0: mean=1.0032 max=1.0096
    R-hat discriminations: mean=1.0046 max=1.0119
    R-hat ddifficulties: mean=1.0033 max=1.0131
    max R-hat overall: 1.0131
  Standardizing marginal: mu=-0.1785, sigma=1.0171
    theta after standardisation: mean=-0.0316 sd=1.0138 mean PSD=0.3855
    saved /home/luis/CGE/mice_imp/results_alt_anchor/standard/nobike/flat.npz

=== noise floor (flat vs birt: same posterior by construction) ===
    theta RMSE       0.00256
    max |delta a|    0.02838
    B-MI vs B-IRT    0.37262  (145.6x the floor)
  wrote /home/luis/CGE/mice_imp/results_alt_anchor/standard/nobike/item_parameters.csv  (60 rows, arms: birt, bmi, flat)
  wrote /home/luis/CGE/mice_imp/results_alt_anchor/standard/nobike/theta.csv  (1363 rows)
  -> ok  (812.4 min)  summary: /home/luis/CGE/mice_imp/results_a


GRID SUMMARY
     design    variant itemset  n_items             started                     status  minutes  birt_max_rhat  bmi_max_rhat  flat_max_rhat  bmi_fallback_pct  advi_init_count  theta_rmse_flat_vs_birt  theta_rmse_bmi_vs_birt
 commonitem alt_anchor    full       27 2026-09-17 09:44:32 skipped (already complete)      0.0       1.017295      1.073667       1.023703               NaN              NaN                 0.002173                0.165862
 commonitem alt_anchor  nobike       20 2026-09-17 09:44:32 skipped (already complete)      0.0       1.008560      1.021609       1.008343               NaN              NaN                 0.001277                0.171501
binterlaced alt_anchor    full       27 2026-09-17 09:44:32                         ok   1211.5       1.017067      1.079643       1.021439               0.0              3.0                 0.003383                0.217713
binterlaced alt_anchor  nobike       20 2026-09-18 05:56:03                         ok    

1

## Progress

Safe to run while the grid is still going: `results_alt_anchor/grid_summary.csv` is
rewritten after every run, and each run's `run.log` is flushed as it goes.

In [3]:
import pandas as pd

summary = pd.read_csv(REPO / "results_alt_anchor" / "grid_summary.csv")
cols = [c for c in ["design", "variant", "itemset", "n_items", "status", "minutes",
                    "birt_max_rhat", "bmi_max_rhat", "flat_max_rhat",
                    "bmi_fallback_pct", "advi_init_count",
                    "theta_rmse_flat_vs_birt", "theta_rmse_bmi_vs_birt"]
        if c in summary.columns]
summary[cols]

,design,variant,itemset,n_items,status,minutes,birt_max_rhat,bmi_max_rhat,flat_max_rhat,bmi_fallback_pct,advi_init_count,theta_rmse_flat_vs_birt,theta_rmse_bmi_vs_birt
0,commonitem,alt_anchor,full,27,skipped (already complete),0.0,1.017295,1.073667,1.023703,NaN,NaN,0.002173,0.165862
1,commonitem,alt_anchor,nobike,20,skipped (already complete),0.0,1.008560,1.021609,1.008343,NaN,NaN,0.001277,0.171501
2,binterlaced,alt_anchor,full,27,ok,1211.5,1.017067,1.079643,1.021439,0.0,3.0,0.003383,0.217713
3,binterlaced,alt_anchor,nobike,20,ok,847.8,1.011208,1.021602,1.008827,0.0,3.0,0.001072,0.406011
4,standard,alt_anchor,full,27,ok,1334.0,1.025657,1.135458,1.038858,0.0,3.0,0.002393,0.162174
5,standard,alt_anchor,nobike,20,ok,812.4,1.010569,1.020774,1.013139,0.0,3.0,0.002559,0.372616


## Re-run one cell

`--only` takes `design/itemset`, e.g. after fixing whatever made a single
run fail. Keep `--variant alt_anchor` on it — without that the re-run reads the
other variant's data and writes over the other variant's results:

```
run_grid("--only", "standard/nobike", "--variant", "alt_anchor", "--force")
```

In [4]:
# run_grid("--only", "standard/nobike", "--variant", "alt_anchor", "--force")